In [ ]:
import os
import json
import glob
from config import DATA_DIR, SCENARIO
import chromadb
from dotenv import load_dotenv
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
import re
from openrouter import OpenRouter
load_dotenv()

d:\agenticaiCapstone\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


False

In [ ]:
CHROMA_DIR="chroma_db"
Memory_DB = "chat_memory.db"
COLLECTION_NAME="pdf_rag_collection"
TOP_K_RETRIEVAL = 10
TOP_K_RERANKED = 4
MEMORY_SUMMARY_EVENTS_N_MESSAGES=5
RECENT_MESSAGES_FOR_CONTEXT=5
chunk_size=400
chunk_overlap=80
PARENT_DIR="d:\\EYAgenticCapstone\\capstone-agenticai-ey"
OUTPUT_PATH=os.path.join(PARENT_DIR,"outputs", "chunks.json")
OpenRouterKey=""


In [4]:

def load_documents(data_dir: str) -> list[dict]:
    """Load every .txt file in data_dir. Returns [{"source": filename, "text": content}]."""
    docs=[]
    for filepath in sorted(glob.glob(os.path.join(data_dir,"*.txt"))):
        with open (filepath,"r",encoding="utf-8") as f:
            text=f.read()
        docs.append({"source":os.path.basename(filepath),"text":text})
    return docs

In [5]:
def get_sentences(para,left_over_window):
    splitter = RecursiveCharacterTextSplitter(separators=["\n",". ","? ","! ","; "],chunk_size=left_over_window,chunk_overlap=0)
    sentences=splitter.split_text(para)
    return sentences

In [6]:
def paragraph_line_sentence_chunking(text,chunk_size,chunk_overlap):
    start_size=0
    chunks=[]
    
    text_len=len(text)
    while start_size<text_len:
        current_size=0
        paras=[]
        add_sentences=[]
        if start_size==0:    
            paragraphs=text[:].split("\n\n")
        else:
            paragraphs=text[start_size:].split("\n\n")
        for para in paragraphs:
            para_len=len(para)
            
            space_window_left=chunk_size-(current_size)
            if para_len<=space_window_left:
                if para_len<=(space_window_left-2):
                    paras.append(para+"\n\n")
                    current_size+=para_len+2

                else:
                    paras.append(para)
                    current_size+=para_len

            else:
                
                sentences=get_sentences(para,left_over_window=space_window_left)
                for sentence in sentences:
                    sentence_len=len(sentence)
                    space_window_left=chunk_size-(current_size)
                    if sentence_len<=space_window_left:
                        if sentence_len<=space_window_left-1:
                            add_sentences.append(sentence+"\n")
                            current_size+=sentence_len+1

                        else:
                            add_sentences.append(sentence)
                            current_size+=sentence_len

                    else:
                        break

                break
        paras.extend(add_sentences)
        add_overlap_text=(text[start_size-chunk_overlap:start_size]) 
        chunk=add_overlap_text+"\n"+"".join(paras) if start_size!=0 else "".join(paras)
        chunks.append(chunk)    
        start_size+=current_size   
           
    return chunks      
                    




In [7]:
def build_chunk_records(docs: list[dict]) -> list[dict]:
    records=[]
    chunk_id=0
    for doc in docs:
        chunks= paragraph_line_sentence_chunking(doc['text'],chunk_size=chunk_size,chunk_overlap=chunk_overlap)
        for chunk in chunks:
            records.append({"chunk_id":chunk_id,
                            "text":chunk,
                            "source":doc['source']})
            chunk_id+=1
    return records

In [8]:
def save_chunks(chunks,OUTPUT_PATH):
    os.makedirs(os.path.dirname(OUTPUT_PATH),exist_ok=True)
    with open(OUTPUT_PATH, "w",encoding="utf-8") as f:
        json.dump(chunks,f,indent=2)
    print("Chunks Saved Successfully")

In [9]:
def ingestion_main():
    raw_docs=load_documents(DATA_DIR)
    print("Documents Loaded")
    records=build_chunk_records(raw_docs)
    print("Chunks Built")
    save_chunks(records,OUTPUT_PATH)
    print("Ingestion Complete")

### Embedding

In [12]:
BASE_DIR = PARENT_DIR
CHUNKS_PATH = os.path.join(BASE_DIR, "outputs", "chunks.json")
CHROMA_DIR = os.path.join(BASE_DIR, "outputs", "chroma_db")
BM25_CORPUS_PATH = os.path.join(BASE_DIR, "outputs", "bm25_corpus.json")
COLLECTION_NAME = f"capstone_{SCENARIO}"
RRF_K=60

In [13]:
def load_chunks():
    with open(CHUNKS_PATH,"r",encoding="utf-8") as f:
        return json.load(f)

In [14]:
chunks=load_chunks()

In [15]:
def generate_embeddings(texts,embedding_model=None):
    if not embedding_model:
        DEFAULT_EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
        embedding_model = SentenceTransformer(DEFAULT_EMBEDDING_MODEL_NAME)
    embeddings=embedding_model.encode(texts,show_progress_bar=False).tolist()
    return embeddings

In [16]:
def create_vectoreStore(chunks,embeddings,vectorDB=None,vectorDB_parameters=None):
    ids=[str(c['chunk_id']) for c in chunks]
    texts=[c['text'] for c in chunks]
    metadatas=[{'source':c['source']} for c in chunks]
    if not vectorDB:
        vectorstore=Chroma(collection_name=COLLECTION_NAME,persist_directory=CHROMA_DIR)
        vectorstore._collection.add(ids=ids,documents=texts,metadatas=metadatas,embeddings=embeddings)

    else:
        vectorstore=vectorDB(*vectorDB_parameters)
    print("vectorstore created sucessfully")
    

In [ ]:


def tokenize(text: str) -> list[str]:
    return re.findall(r"[a-zA-Z0-9]+(?:[-_][a-zA-Z0-9]+)*", text.lower())

In [18]:
def build_bm25_index(chunks):
    token_corpus=[tokenize(c['text']) for c in chunks]
    payload={"ids":[c["chunk_id"] for c in chunks],
             "token_corpus":token_corpus}
    with open(BM25_CORPUS_PATH,"w",encoding="utf-8") as f:
        json.dump(payload,f,indent=2)
    bm25 = BM25Okapi(token_corpus)
    print(f"BM25 index built OK {len(token_corpus)} chunks")
    return bm25




In [19]:
def embed_main():
    chunks=load_chunks()
    print(f"{len(chunks)} Chunks Loaded")
    texts=[c['text'] for c in chunks]
    embeddings=generate_embeddings(texts)
    print("embeddings generated")
    create_vectoreStore(chunks=chunks,embeddings=embeddings)
    print("vectorStore Created")
    bm25Index=build_bm25_index(chunks=chunks)
    print("embedding part done!")
    return bm25Index


In [ ]:
# bm25Index=embed_main()

23 Chunks Loaded
embeddings generated
vectorStore Created
BM25 index built OK 23 chunks
embedding part done!


### Retrieval

In [ ]:
# Run Vector Search
# Run BM25 Search
# Apply Hybrid Search (RRF)
# Apply Reranking



In [20]:
def vectorSearch(query,embedding_model=None,vectorstore=None):
    query_embeddings=generate_embeddings(query,embedding_model=embedding_model)
    if not vectorstore:
        Client=chromadb.PersistentClient(CHROMA_DIR)
        collection=Client.get_collection(COLLECTION_NAME)
    vector_retrival=collection.query(query_embeddings,n_results=TOP_K_RETRIEVAL)
    return [int(i) for i in vector_retrival['ids'][0]]

In [21]:
def BM25Search(query,bm25Index,chunk_ids_list=None):
    tokenized_query=tokenize(query)
    bm25_scores=bm25Index.get_scores(tokenized_query)
    ids=chunk_ids_list
    if not ids:
        ids=[i for i in range(len(bm25_scores))]
    ranked=[i for i,j in sorted(zip(ids,bm25_scores),key=lambda x: x[1],reverse=True)]
    return ranked

In [22]:
def reciprocal_rank_fusion(ranked_lists: list[list[str]], k: int = RRF_K) -> list[str]:
    scores={}
    for ranked_list in ranked_lists:
        for rank,chunk_id in enumerate(ranked_list):
            scores[chunk_id]=scores.get(chunk_id,0)+1/(rank+k+1)
    fused_ranks=sorted(scores.items(),key=lambda x:x[1],reverse=True )
    return [chunk_id for chunk_id,score in fused_ranks]

In [23]:
def retrieval_main(query: str,bm25Index,embedding_model=None,vectorstore=None,chunk_ids_list=None,top_k_retrieval=TOP_K_RETRIEVAL):
    chunks=load_chunks()
    chunk_lookup={chunk["chunk_id"]:chunk for chunk in chunks}

    vector_retrieval_ids=vectorSearch(query,embedding_model=embedding_model,vectorstore=vectorstore)[:top_k_retrieval]

    bm25_retrieval_ids=BM25Search(query,bm25Index=bm25Index,chunk_ids_list=chunk_ids_list)[:top_k_retrieval]
  
    hybrid_rank_ids=reciprocal_rank_fusion(ranked_lists=[vector_retrieval_ids,bm25_retrieval_ids],k=RRF_K)[:top_k_retrieval]
 
    retrieved_chunks=[chunk_lookup[chunk_id] for chunk_id in hybrid_rank_ids if chunk_id in chunk_lookup]
    return retrieved_chunks


### Generation

In [25]:
def build_prompt(query,retrieved_chunks,chat_history=""):
    persona=""
    if SCENARIO.lower() == "banking":
        persona="""You are a helpful, precise customer support assistant for a retail bank. Your scope is to answer customer questions about  loans, fees, accounts and fraud policy ONLY. You are not supposed to take any action from bank's side"""
    context_block="\n\n".join([f"[Source: {chunk['source']}]\ntext: {chunk['text']}" for chunk in retrieved_chunks])[:]
    system_prompt=f"""{persona}
STRICT INSTRUCTIONS:
- Answer only based on the context provided to you.
- If the context does not contain enough information to answer confidently,
explicitly say ("I donot have enough information to answer precisely - please contact support team directly")
rather than guessing.
- Always mention the source files you used at the end of your answer, like:
"Sources: loan_policy.txt, faq_emi.txt"
- Keep the answers concise: 2-5 sentences plus the source citation line.
- As you are a helpful banking customer support assistant you should be polite and be welcoming for any questions from user.
- If user sends a general greeting, repond with greetings as if you are actual assistant. ( you need not mention source in such cases)

    """
    user_prompt=f"""Here is the chat history:{chat_history}
    Here is retrieved context based on user query: 
Retrieved Context: 
{context_block}


User: {query}"""
    return [{"role":"system","content":system_prompt},{"role":"user","content":user_prompt}]


In [ ]:
def call_llm(messages,temperature,max_tokens):
    
    with OpenRouter(
          api_key=os.getenv("OPENROUTER_API_KEY", OpenRouterKey),
        ) as client:
                response = client.chat.send(model="nvidia/nemotron-3-super-120b-a12b:free",
                messages=messages,
                temperature=temperature,
                max_tokens=max_tokens)
        
    return response.choices[0].message.content

In [27]:
def generate_answer_main(query,retrieved_chunks,chat_history=None):
    messages=build_prompt(query,retrieved_chunks,chat_history)
    answer=call_llm(messages=messages,temperature=0.1,max_tokens=1500)
    return answer

### RAG MAIN

In [28]:
def rag_main(query):
   ingestion_main()
   print("ingestion Complete")
   bm25Index=embed_main()
   print("embedding Complete")
   retrieved_chunks=retrieval_main(query,bm25Index=bm25Index)
   print("Retrieval Complete")
   answer=generate_answer_main(query,retrieved_chunks)
   print("answer generated")
   return answer,bm25Index

In [29]:
query="I want to apply for loan"
ans,bm25Index=rag_main(query)

Documents Loaded
Chunks Built
Chunks Saved Successfully
Ingestion Complete
ingestion Complete
23 Chunks Loaded
embeddings generated


C:\Users\hp\AppData\Local\Temp\ipykernel_15076\3184632730.py:6: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 0.4. An updated version of the class exists in the langchain-chroma package and should be used instead. To use it run `pip install -U langchain-chroma` and import as `from langchain_chroma import Chroma`.
  vectorstore=Chroma(collection_name=COLLECTION_NAME,persist_directory=CHROMA_DIR)
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


vectorstore created sucessfully
vectorStore Created
BM25 index built OK 23 chunks
embedding part done!
embedding Complete


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Retrieval Complete
answer generated


In [30]:
ans

"Hello! I'd be happy to help you understand how to apply for a personal loan. Based on our policies, here's what you need to know:\n\n**Eligibility Requirements:**\n- Age: 21 to 60 years\n- Income: Minimum INR 25,000 monthly net income for salaried applicants OR annual turnover of INR 12,00,000 for self-employed applicants\n- Credit Score: Minimum 700 for standard approval (scores 650-699 undergo secondary review)\n\n**Loan Details:**\n- Loan amounts range from INR 50,000 to INR 25,00,000\n- Repayment tenure: 12 to 60 months\n- Maximum loan amount is capped at 20 times the applicant's average monthly net income\n- Interest rates: 10.5% to 18% per annum (based on credit score, income stability, and debt-to-income ratio)\n\n**Application Process:**\n- Applications remain open for 90 days from submission\n- You can request instalment deferral (up to 2 EMIs per loan tenure) for genuine financial hardship, subject to approval\n- Late payments incur a charge of 2% of the overdue amount plus 

### ChatBot

In [31]:
def generate_query(query,chat_history):
    """Drafts a new query which can be used to fetch out the relevant documents from database
    Parameters: 
    user_query: what user is asking currently (based on past interaction).
   """
    prompt=[{"role":"system",
             "content":
             f"""You are an assistant to customer care agent for a retail bank.
You are an expert in drafting queries so that most relevant information can be obtained when performed hybrid search (semantic and keyword).
You would be given the user query and chat_history between user and the agent.
You need to draft new query, which would have high probability to semantically relate to required documents of text.
At the same time You may also include specific domain keywords in new query so that search of document become even better with keyword based search.
**IMPORTANT**
- If there is no chat history present assume, it is start of chat.
- Your drafted query should be atleast 20 words (and at most 50 words), such that it becomes easier to search relevant data using your query.
- You should never reply back as None or Null
- If consider query to be good enough then return the same as it is.
- Do Not include customer's name in your generated query
- Do Not overcomplicate the query, understand the intent from chat history and user query and accordingly respond.

Your output should be nothing else than following format.
"Query: drafted_query"
"""},
{"role":"user","content":f"""Here is the current user_query: {query}\nChat History: {chat_history}
"""}]
    drafted_query=call_llm(messages=prompt,temperature=1,max_tokens=1000)
    return drafted_query
    

In [32]:
def memory_summarizer_agent(chat_history):
    system_prompt="""You are a Chat Summarizer Agent to a "Customer Support Agent for a retail bank". 
    Your role is to summarize the chat_history between user and Support Agent,
    such that only the meaningful parts of the chat get consolidated in a brief and precise piece of text.
    While generating the summary do remeber to preserve user PII, intent and user preferneces.
    The summary generated by you should help the Support agent in following manner:
    - Support Agent need not to read the whole historical chat interaction, he just gets to know about whole chat and user intent by reading just a few lines of text (that is your generated summary).
    - Just the meaningful parts of the chat (which are required by Support Agent for accomplishing his objective of providing best service) is preserved filtering out any other generic stuff like greetings, wishes, misunderstandings, etc.

    Your answer should strictly be just the following:
    "summarized memory: (the generated summary)"
"""
    user_prompt=f"""Kindly summarize the following chat history between Customer Support Agent and User: 
    {chat_history}"""
    messages=[{"role":"system","content":system_prompt},{"role":"user","content":user_prompt}]
    generated_summary=call_llm(messages=messages,temperature=0.5,max_tokens=5000)
    return generated_summary
    

In [33]:
def chatBot():
    user_query = ""
    chat_history = []
    while True:
        user_query = input("Your next query please (Type quit to exit): ")
        print("user: ",user_query)
        # termination condition
        if user_query == "quit": 
            break
        drafted_query = generate_query(user_query,chat_history=chat_history)
        print("Drafted Query: ",drafted_query)
        retrieved_chunks = retrieval_main(drafted_query,bm25Index,top_k_retrieval=4)
        answer = generate_answer_main(user_query,retrieved_chunks=retrieved_chunks,chat_history=chat_history)
        print("Support Agent: ",answer)
        chat_history.extend([f"user : {user_query}",f"Support Agent: {answer}"])
        if len(chat_history)>=8:
            summarized_chat=memory_summarizer_agent(chat_history[:-2]).strip()
            summarized_chat=[summarized_chat.removeprefix("summarized memory:").removeprefix("Summarized memory")]
            print("summarized chat history: ",summarized_chat)
            summarized_chat.extend(chat_history[-2:])
            chat_history=summarized_chat
        print("Chat History: ",chat_history)
    return chat_history
        



In [36]:
chatBot() # enter quit to end

user:  Hi I am Ava, I am 22 years
Drafted Query:  Query: banking products and services suitable for 22 year old young adults, including checking accounts, savings options, low-fee solutions, student benefits, and introductory offers for first-time bank customers


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Support Agent:  Hello Ava! Welcome to our retail banking support. How can I assist you today?
Chat History:  ['user : Hi I am Ava, I am 22 years', 'Support Agent: Hello Ava! Welcome to our retail banking support. How can I assist you today?']
user:   I want apply a loan of 100000. Can I get it?
Drafted Query:  Query: eligibility criteria for a personal loan of 100,000 for a 22 year old applicant required documents interest rate approval process minimum income credit score


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Support Agent:  Based on the loan policy, a personal loan of INR 100,000 falls within the allowed range of INR 50,000 to INR 25,00,000. However, approval also depends on meeting eligibility criteria: you must be aged 21‑60, have a minimum monthly net income of INR 25,000 (if salaried) or the equivalent turnover (if self‑employed), and a credit score of at least 700. Since we don’t have your income, employment type, or credit‑score details, I cannot confirm whether you qualify. Please provide those details or contact the support team for a precise assessment.  

Sources: loan_policy.txt
Chat History:  ['user : Hi I am Ava, I am 22 years', 'Support Agent: Hello Ava! Welcome to our retail banking support. How can I assist you today?', 'user :  I want apply a loan of 100000. Can I get it?', 'Support Agent: Based on the loan policy, a personal loan of INR\u202f100,000 falls within the allowed range of INR\u202f50,000 to INR\u202f25,00,000. However, approval also depends on meeting eligibili

...

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Support Agent:  Based on the loan policy, personal‑loan interest rates range from **10.5% to 18% per annum**, with the exact rate depending on your credit score, income stability, and debt‑to‑income ratio. Given your credit score of 900 and stable monthly income of INR 30,000, you would likely qualify for the lower end of that range, but the final rate will be confirmed during the loan assessment.  

Sources: loan_policy.txt
summarized chat history:  [' Ava, 22, inquired about eligibility for an INR\u202f100,000 personal loan, providing a stable monthly income of INR\u202f30,000 and a credit score of 900. The support agent confirmed she meets the loan policy requirements—age 21‑60, income above the INR\u202f25,000 minimum, credit score ≥700, loan amount within the allowed range (INR\u202f50,000–INR\u202f25,00,000) and below the 20× income cap—and stated she is eligible, subject to final document verification.']
Chat History:  [' Ava, 22, inquired about eligibility for an INR\u202f100,0

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Support Agent:  Based on the loan policy, personal‑loan interest rates range from **10.5% to 18% per annum**, with the exact rate depending on your credit score, income stability, and debt‑to‑income ratio. Given your credit score of 900 and stable monthly income of INR 30,000, you are likely to qualify for the lower end of that range, so a rate around **12% is possible**, but the final rate will be confirmed during the loan assessment.  

Sources: loan_policy.txt
Chat History:  [' Ava, 22, inquired about eligibility for an INR\u202f100,000 personal loan, providing a stable monthly income of INR\u202f30,000 and a credit score of 900. The support agent confirmed she meets the loan policy requirements—age 21‑60, income above the INR\u202f25,000 minimum, credit score ≥700, loan amount within the allowed range (INR\u202f50,000–INR\u202f25,00,000) and below the 20× income cap—and stated she is eligible, subject to final document verification.', 'user : what would be the interest rates for me

[' Ava, 22, inquired about eligibility for an INR\u202f100,000 personal loan, providing a stable monthly income of INR\u202f30,000 and a credit score of 900. The support agent confirmed she meets the loan policy requirements—age 21‑60, income above the INR\u202f25,000 minimum, credit score ≥700, loan amount within the allowed range (INR\u202f50,000–INR\u202f25,00,000) and below the 20× income cap—and stated she is eligible, subject to final document verification.',
 'user : what would be the interest rates for me?',
 'Support Agent: Based on the loan policy, personal‑loan interest rates range from **10.5% to 18% per annum**, with the exact rate depending on your credit score, income stability, and debt‑to‑income ratio. Given your credit score of 900 and stable monthly income of INR\u202f30,000, you would likely qualify for the lower end of that range, but the final rate will be confirmed during the loan assessment.  \n\nSources: loan_policy.txt',
 'user : will I get it at interest rate